# 外部データの場所
https://console.cloud.google.com/storage/browser/qwiklabs-asl-00-8534302ca246-data;tab=objects?forceOnBucketsSortingFiltering=true&project=qwiklabs-asl-00-8534302ca246&prefix=&forceOnObjectsSortingFiltering=false

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [3]:
from google.cloud import aiplatform

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_supp

In [4]:
REGION = "us-central1"
PROJECT_ID = !(gcloud config get-value project)
PROJECT_ID = PROJECT_ID[0]

In [5]:
!cat trainer_image_vertex/Dockerfile

FROM us-docker.pkg.dev/vertex-ai/training/sklearn-cpu.1-0:latest
RUN pip install fire cloudml-hypertune pandas scikit-learn==1.2.2

# WORKDIR /app
WORKDIR /app

COPY train.py .
ENTRYPOINT ["python", "train.py"]

In [6]:
ARTIFACT_REGISTRY_DIR = "asl-artifact-repo"
IMAGE_NAME = "trainer_image_covertype_vertex"
# IMAGE_TAG = "latest"
IMAGE_TAG = "V8"
TRAINING_CONTAINER_IMAGE_URI = f"us-docker.pkg.dev/{PROJECT_ID}/{ARTIFACT_REGISTRY_DIR}/{IMAGE_NAME}:{IMAGE_TAG}"
TRAINING_CONTAINER_IMAGE_URI

'us-docker.pkg.dev/qwiklabs-asl-00-8534302ca246/asl-artifact-repo/trainer_image_covertype_vertex:V8'

In [7]:
!gcloud builds submit --timeout 15m --tag $TRAINING_CONTAINER_IMAGE_URI trainer_image_vertex

Creating temporary archive of 8 file(s) totalling 20.6 KiB before compression.
Uploading tarball of [trainer_image_vertex] to [gs://qwiklabs-asl-00-8534302ca246_cloudbuild/source/1772978691.421305-1fa1788439fc44509f12b557b5d105e8.tgz]
Created [https://cloudbuild.googleapis.com/v1/projects/qwiklabs-asl-00-8534302ca246/locations/global/builds/a6fc7e35-4924-42fe-8882-ef5e909e9a84].
Logs are available at [ https://console.cloud.google.com/cloud-build/builds/a6fc7e35-4924-42fe-8882-ef5e909e9a84?project=998865242242 ].
Waiting for build to complete. Polling interval: 1 second(s).
----------------------------- REMOTE BUILD OUTPUT ------------------------------
starting build "a6fc7e35-4924-42fe-8882-ef5e909e9a84"

FETCHSOURCE
Fetching storage object: gs://qwiklabs-asl-00-8534302ca246_cloudbuild/source/1772978691.421305-1fa1788439fc44509f12b557b5d105e8.tgz#1772978691674353
Copying gs://qwiklabs-asl-00-8534302ca246_cloudbuild/source/1772978691.421305-1fa1788439fc44509f12b557b5d105e8.tgz#1772978

In [8]:
SERVING_CONTAINER_IMAGE_URI = (
    # 0308 Scikit-learn 1.2対応
    # "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"
    "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-2:latest"
)

In [9]:
# GCS上のファイルパス（gs://バケット名/ファイル名）
file_path = "gs://qwiklabs-asl-00-8534302ca246-data/ENB2012_data.xlsx"

# pandasで読み込み
df = pd.read_excel(file_path)

# データの先頭を確認
print(df.head())

     X1     X2     X3      X4   X5  X6   X7  X8     Y1     Y2
0  0.98  514.5  294.0  110.25  7.0   2  0.0   0  15.55  21.33
1  0.98  514.5  294.0  110.25  7.0   3  0.0   0  15.55  21.33
2  0.98  514.5  294.0  110.25  7.0   4  0.0   0  15.55  21.33
3  0.98  514.5  294.0  110.25  7.0   5  0.0   0  15.55  21.33
4  0.90  563.5  318.5  122.50  7.0   2  0.0   0  20.84  28.28


In [10]:
# dfをトレーニング用(train)と評価用(test)に分割
# test_size=0.2 は全体の20%を評価用にすることを意味します
# random_state=42 は分割のされ方を固定するための値です（数値は何でもOK）
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# 分割後のデータ数を確認
print(f"トレーニング用データ数: {len(train_df)}")
print(f"評価用データ数: {len(test_df)}")

トレーニング用データ数: 614
評価用データ数: 154


In [11]:
# 特徴量(X)とターゲット(Y1)の選択
X = df.drop(columns=['Y1', 'Y2']) # Y1とY2以外をすべて入力データにする
y = df['Y1']                      # Y1を予測対象にする

# 4つの変数に一気に分割
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 確認
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (614, 8)
X_test shape: (154, 8)


In [12]:
%%writefile ./pipeline_vertex/pipeline.py
# Copyright 2021 Google LLC

# Licensed under the Apache License, Version 2.0 (the "License"); you may not
# use this file except in compliance with the License. You may obtain a copy of
# the License at

# https://www.apache.org/licenses/LICENSE-2.0

# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS"
# BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either
# express or implied. See the License for the specific language governing
# permissions and limitations under the License.
"""Kubeflow Covertype Pipeline."""
import os

from kfp import dsl
from training_lightweight_component import train_and_deploy
from tuning_lightweight_component import tune_hyperparameters

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
PROJECT_ID = os.getenv("PROJECT_ID")
REGION = os.getenv("REGION")

TRAINING_CONTAINER_IMAGE_URI = os.getenv("TRAINING_CONTAINER_IMAGE_URI")
SERVING_CONTAINER_IMAGE_URI = os.getenv("SERVING_CONTAINER_IMAGE_URI")

TRAINING_FILE_PATH = os.getenv("TRAINING_FILE_PATH")
VALIDATION_FILE_PATH = os.getenv("VALIDATION_FILE_PATH")

# MAX_TRIAL_COUNT = int(os.getenv("MAX_TRIAL_COUNT", "5"))
# PARALLEL_TRIAL_COUNT = int(os.getenv("PARALLEL_TRIAL_COUNT", "5"))
MAX_TRIAL_COUNT = int(os.getenv("MAX_TRIAL_COUNT", "10")) #最大試行回数
PARALLEL_TRIAL_COUNT = int(os.getenv("PARALLEL_TRIAL_COUNT", "2")) #同時にいくつまで学習ジョブを走らせるか　(最大試行回数の10%程度が目安）
THRESHOLD = float(os.getenv("THRESHOLD", "0.6")) #デプロイの閾値 →これを越えないとデプロイまで流れない


@dsl.pipeline(
    name="covertype-kfp-pipeline_test",
    description="The pipeline training and deploying the Covertype classifier",
    pipeline_root=PIPELINE_ROOT,
)
def covertype_train(
    training_container_uri: str = TRAINING_CONTAINER_IMAGE_URI,
    serving_container_uri: str = SERVING_CONTAINER_IMAGE_URI,
    training_file_path: str = TRAINING_FILE_PATH,
    validation_file_path: str = VALIDATION_FILE_PATH,
    # accuracy_deployment_threshold: float = THRESHOLD,
    r2_score_deployment_threshold: float = THRESHOLD,
    max_trial_count: int = MAX_TRIAL_COUNT,
    parallel_trial_count: int = PARALLEL_TRIAL_COUNT,
    pipeline_root: str = PIPELINE_ROOT,
):
    staging_bucket = f"{pipeline_root}/staging"

    tuning_op = tune_hyperparameters(
        project=PROJECT_ID,
        location=REGION,
        container_uri=training_container_uri,
        training_file_path=training_file_path,
        validation_file_path=validation_file_path,
        staging_bucket=staging_bucket,
        max_trial_count=max_trial_count,
        parallel_trial_count=parallel_trial_count,
    )

    # accuracy = tuning_op.outputs["best_accuracy"]
    r2_score = tuning_op.outputs["best_r2_score"]

    with dsl.If(
        # accuracy >= accuracy_deployment_threshold, name="deploy_decision"
        r2_score >= r2_score_deployment_threshold, name="deploy_decision"
    ):
        train_and_deploy_op = (  # pylint: disable=unused-variable
            train_and_deploy(
                project=PROJECT_ID,
                location=REGION,
                container_uri=training_container_uri,
                serving_container_uri=serving_container_uri,
                training_file_path=training_file_path,
                validation_file_path=validation_file_path,
                staging_bucket=staging_bucket,
                alpha=tuning_op.outputs["best_alpha"],
                max_iter=tuning_op.outputs["best_max_iter"],
            )
        )

Overwriting ./pipeline_vertex/pipeline.py


In [13]:
# # ノートブック上の定義を回帰用に変更（例：誤差10.0以下でデプロイ）
# THRESHOLD = 10.0 

# @dsl.pipeline(
#     name="regression-kfp-pipeline",
#     description="回帰モデルを訓練してデプロイするパイプライン",
#     pipeline_root=PIPELINE_ROOT,
# )
# def covertype_train(
#     training_container_uri: str = TRAINING_CONTAINER_IMAGE_URI,
#     serving_container_uri: str = SERVING_CONTAINER_IMAGE_URI,
#     training_file_path: str = TRAINING_FILE_PATH,
#     validation_file_path: str = VALIDATION_FILE_PATH,
#     # 引数名を精度(accuracy)から誤差(rmse)に変更
#     rmse_deployment_threshold: float = THRESHOLD,
#     max_trial_count: int = MAX_TRIAL_COUNT,
#     parallel_trial_count: int = PARALLEL_TRIAL_COUNT,
#     pipeline_root: str = PIPELINE_ROOT,
# ):
#     staging_bucket = f"{pipeline_root}/staging"

#     tuning_op = tune_hyperparameters(
#         project=PROJECT_ID,
#         location=REGION,
#         container_uri=training_container_uri,
#         training_file_path=training_file_path,
#         validation_file_path=validation_file_path,
#         staging_bucket=staging_bucket,
#         max_trial_count=max_trial_count,
#         parallel_trial_count=parallel_trial_count,
#     )

#     # tuning_opの出力から、最も良かった(小さかった)RMSEを取得
#     # ※tune_hyperparametersコンポーネント内のキー名と一致させてください
#     rmse = tuning_op.outputs["best_rmse"]

#     # 比較演算子を「以下(<=)」に変更（誤差が閾値より小さければデプロイ）
#     with dsl.If(
#         rmse <= rmse_deployment_threshold, name="deploy_decision"
#     ):
#         train_and_deploy_op = (
#             train_and_deploy(
#                 project=PROJECT_ID,
#                 location=REGION,
#                 container_uri=training_container_uri,
#                 serving_container_uri=serving_container_uri,
#                 training_file_path=training_file_path,
#                 validation_file_path=validation_file_path,
#                 staging_bucket=staging_bucket,
#                 alpha=tuning_op.outputs["best_alpha"],
#                 max_iter=tuning_op.outputs["best_max_iter"],
#             )
#         )


# 重要：コンポーネント側の修正
# この pipeline.py を変更するだけでは不十分です。以下の2つのファイルも回帰（Regression）用に修正してください。
# train.py (学習スクリプト):

# モデルを RandomForestClassifier から RandomForestRegressor などに変更。

# 評価関数を accuracy_score から mean_squared_error に変更。

# hypertune ライブラリに報告する指標名を rmse などに揃える。

# tuning_lightweight_component.py:

# Outputs の定義で best_accuracy となっている箇所を best_rmse に変更。





In [14]:
ARTIFACT_STORE = f"gs://{PROJECT_ID}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATA_ROOT = f"{ARTIFACT_STORE}/data"

TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT_ID={PROJECT_ID}
%env REGION={REGION}
%env SERVING_CONTAINER_IMAGE_URI={SERVING_CONTAINER_IMAGE_URI}
%env TRAINING_CONTAINER_IMAGE_URI={TRAINING_CONTAINER_IMAGE_URI}
%env TRAINING_FILE_PATH={TRAINING_FILE_PATH}
%env VALIDATION_FILE_PATH={VALIDATION_FILE_PATH}

env: PIPELINE_ROOT=gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/pipeline
env: PROJECT_ID=qwiklabs-asl-00-8534302ca246
env: REGION=us-central1
env: SERVING_CONTAINER_IMAGE_URI=us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-2:latest
env: TRAINING_CONTAINER_IMAGE_URI=us-docker.pkg.dev/qwiklabs-asl-00-8534302ca246/asl-artifact-repo/trainer_image_covertype_vertex:V8
env: TRAINING_FILE_PATH=gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/data/training/dataset.csv
env: VALIDATION_FILE_PATH=gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/data/validation/dataset.csv


In [15]:
# 分割したデータをCSVとして一時保存
train_df.to_csv("train_data.csv", index=False)
test_df.to_csv("test_data.csv", index=False)

# 環境変数（以前定義したもの）を使用してGCSへコピー
# DATA_ROOT = f"gs://{PROJECT_ID}-kfp-artifact-store/data" と仮定
!gsutil cp train_data.csv {DATA_ROOT}/training/dataset.csv
!gsutil cp test_data.csv {DATA_ROOT}/validation/dataset.csv

# パイプラインが参照するパスを確認
TRAINING_FILE_PATH = f"{DATA_ROOT}/training/dataset.csv"
VALIDATION_FILE_PATH = f"{DATA_ROOT}/validation/dataset.csv"

Copying file://train_data.csv [Content-Type=text/csv]...
/ [1 files][ 28.3 KiB/ 28.3 KiB]                                                
Operation completed over 1 objects/28.3 KiB.                                     
Copying file://test_data.csv [Content-Type=text/csv]...
/ [1 files][  7.1 KiB/  7.1 KiB]                                                
Operation completed over 1 objects/7.1 KiB.                                      


In [16]:
!gsutil ls | grep ^{ARTIFACT_STORE}/$ || gsutil mb -l {REGION} {ARTIFACT_STORE}

gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/


In [17]:
PIPELINE_YAML = "covertype_kfp_pipeline.yaml"

In [18]:
!kfp dsl compile --py pipeline_vertex/pipeline.py --output $PIPELINE_YAML

Pipeline code was successfully compiled with the output saved to /home/jupyter/covertype_kfp_pipeline.yaml


In [19]:
!head {PIPELINE_YAML}

# PIPELINE DEFINITION
# Name: covertype-kfp-pipeline-test
# Description: The pipeline training and deploying the Covertype classifier
# Inputs:
#    max_trial_count: int [Default: 10.0]
#    parallel_trial_count: int [Default: 2.0]
#    pipeline_root: str [Default: 'gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/pipeline']
#    r2_score_deployment_threshold: float [Default: 0.6]
#    serving_container_uri: str [Default: 'us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-2:latest']
#    training_container_uri: str [Default: 'us-docker.pkg.dev/qwiklabs-asl-00-8534302ca246/asl-artifact-repo/trainer_image_covertype_vertex:V8']


In [20]:
aiplatform.init(project=PROJECT_ID, location=REGION)

pipeline = aiplatform.PipelineJob(
    display_name="covertype_kfp_pipeline",
    template_path=PIPELINE_YAML,
    enable_caching=False,
)

In [21]:
pipeline.run()

Creating PipelineJob
PipelineJob created. Resource name: projects/998865242242/locations/us-central1/pipelineJobs/covertype-kfp-pipeline-test-20260308140718
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/998865242242/locations/us-central1/pipelineJobs/covertype-kfp-pipeline-test-20260308140718')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/covertype-kfp-pipeline-test-20260308140718?project=998865242242
PipelineJob projects/998865242242/locations/us-central1/pipelineJobs/covertype-kfp-pipeline-test-20260308140718 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/998865242242/locations/us-central1/pipelineJobs/covertype-kfp-pipeline-test-20260308140718 current state:
PipelineState.PIPELINE_STATE_RUNNING
PipelineJob projects/998865242242/locations/us-central1/pipelineJobs/covertype-kfp-pipeline-test-20260308140718 current state:
PipelineState.PIPELINE_STATE_RUNNING

In [22]:
!gsutil ls {TRAINING_FILE_PATH}

gs://qwiklabs-asl-00-8534302ca246-kfp-artifact-store/data/training/dataset.csv
